# Classification Comparison to Persistence Landscapes and Persistence Images under LOOCV and Bootstrap subsampling

In [1]:
# load packages and data
import numpy as np
import pickle
import math
import time
import sys
# Assuming utils_load_PHloc is accessible or its content is in the same file
from utils_load_PHloc import datasets_of_interest, injected_datasets_of_interest, levels_of_interest
import matplotlib.pyplot as plt
import sklearn
import torch
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pandas as pd
from sklearn.utils import resample
from tqdm import tqdm
from sklearn.model_selection import cross_validate
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer, precision_recall_fscore_support
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from persim.landscapes import PersistenceLandscaper
from persim import plot_diagrams
from warnings import simplefilter
from gtda.diagrams import PersistenceImage, PersistenceLandscape
import gc

# Suppress ConvergenceWarning from sklearn.svm._base.py
simplefilter(action='ignore', category=FutureWarning)

PH_folder = '' 
anatomy = 'knee'
filepath = PH_folder + 'PH_all_{}.pkl'.format(anatomy)
PH_all_datasets = pickle.load(open(filepath, 'rb'))

# threshold value
THR = .5
# generate a new dictionary for truncated datasets
truncated_PH_all_datasets = {}
for i in datasets_of_interest:
    diagram = PH_all_datasets[i]
    truncated_PH_all_datasets[i] = diagram[diagram[:,1] >= diagram[:,0] + THR]
    
names = [injected_datasets_of_interest[i]+"_"+[str(x) for x in datasets_of_interest][i] for i in range(27)]

labels = ['CTRL_0%(1)', 'CTRL_0%(2)', 'CTRL_0%(3)', 'CTRL_0%(4)', 
          'U937_1%(1)', 'U937_1%(2)', 'U937_7%', 'U937_8%', 'U937_10%(1)', 'U937_10%(2)', 'U937_10%(3)', 
          'HL60_23%', 'HL60_25%(1)', 'HL60_25%(2)', 
          'P1_10%', 'P1_40%', 'P1_44%', 'P1_51%', 'P1_60%', 'P1_76%', 
          'P2_59%', 'P2_88%', 'P2_90%', 
          'MNC_53%', 'MNC_67%', 'MNC_75%', 'MNC_86%']

phases_of_interest = [0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,2,
         2,2,2,
         2,2,2,2]

name_phase = [labels[i]+" [Phase "+str(phases_of_interest[i])+"]" for i in range(27)]

# Binning parameters
XLIMS = np.array([[-15,0],[-10,10],[0,20]])
YLIMS = np.array([[-8,7],[-5,15],[0,20]])
NB_BINS_PER_SIDE = 100

pds = [truncated_PH_all_datasets[i] for i in datasets_of_interest]

[pyKeOps]: Warning, no cuda detected. Switching to cpu only.


In [2]:
# --- Helper functions for padding diagrams ---
def pad_diagram_all_dims(dgm, max_pts_per_dim):
    """Pads a single diagram across all dimensions to target lengths."""
    padded_dgm_parts = []
    # Determine the unique dimensions present in the diagram
    present_dims = np.unique(dgm[:, 2]).astype(int)

    for dim_idx in range(3): # Iterate over homology dimensions 0, 1, 2
        # Select only the rows with the current homology dimension
        dgm_dim = dgm[dgm[:, 2] == dim_idx]
        target_len = max_pts_per_dim[dim_idx]

        pad_len = target_len - dgm_dim.shape[0]
        if pad_len > 0:
            # Pad with (0.0, 0.0, dim_idx) points
            # It's important to keep the correct dimension index for padded points
            pad = np.array([[0.0, 0.0, dim_idx]] * pad_len)
            dgm_dim = np.vstack([dgm_dim, pad])
        padded_dgm_parts.append(dgm_dim)

    # Filter out empty parts if a dimension was not present and had 0 max_pts_per_dim
    # This ensures vstack doesn't get empty lists if a diagram *type* had no points
    return np.vstack([part for part in padded_dgm_parts if part.shape[0] > 0])


def pad_diagrams(diags):
    """
    Pads a list of diagrams so that each diagram has the same number of points
    per homology dimension.
    """
    if not diags:
        return np.array([])

    # Calculate max points per dimension across all diagrams
    max_pts_per_dim = [0, 0, 0] # For dimensions 0, 1, 2
    for dgm in diags:
        if dgm.shape[0] == 0: continue # Skip empty diagrams
        for dim_idx in range(3):
            count = np.sum(dgm[:, 2] == dim_idx)
            if count > max_pts_per_dim[dim_idx]:
                max_pts_per_dim[dim_idx] = count

    # Apply padding to each diagram
    padded_diags = []
    for dgm in diags:
        # If max_pts_per_dim is [0,0,0] (all diagrams are empty for some dimension)
        # or if the current diagram is empty, create a dummy diagram for padding purposes
        if dgm.shape[0] == 0:
            # Create a dummy diagram that has at least one point for each non-zero max_pts_per_dim
            dummy_dgm = np.empty((0,3))
            for dim_idx, max_len in enumerate(max_pts_per_dim):
                if max_len > 0:
                    dummy_dgm = np.vstack([dummy_dgm, np.array([[0.0, 0.0, dim_idx]])])
            if dummy_dgm.shape[0] == 0: # If all max_pts are 0, just add an empty 3D array
                 padded_diags.append(np.empty((0,3)))
            else:
                 padded_diags.append(pad_diagram_all_dims(dummy_dgm, max_pts_per_dim))
        else:
            padded_diags.append(pad_diagram_all_dims(dgm, max_pts_per_dim))
    return np.array(padded_diags)


# Helper functions for filtering diagrams by dimension or quadrant
def filter_diagrams_by_dimension(diagrams_list, target_dim):
    """Filters a list of diagrams to keep only points of a specific dimension."""
    filtered_list = []
    for dgm in diagrams_list:
        if dgm.shape[0] == 0: # Handle empty input diagrams
            filtered_list.append(np.empty((0, 3)))
            continue
        filtered_list.append(dgm[dgm[:, 2] == target_dim])
    return filtered_list

def filter_diagrams_by_quadrant(diagrams_list, x_cond_func, y_cond_func, source_dim):
    """
    Filters a list of diagrams (assumed to be already dimension-specific)
    by birth-death quadrant conditions.
    source_dim is the original dimension, kept for the 3rd column of the filtered points.
    """
    filtered_list = []
    for dgm in diagrams_list:
        if dgm.shape[0] == 0:
            filtered_list.append(np.empty((0, 3)))
            continue
        x = dgm[:, 0]
        y = dgm[:, 1]
        # Ensure the 3rd column (dimension) is correctly maintained for filtered points
        filtered_points = dgm[(x_cond_func(x)) * (y_cond_func(y))]
        filtered_list.append(filtered_points)
    return filtered_list



In [3]:
y_labels = np.array(phases_of_interest)

In [4]:
# --- REGIME 2: LOOCV with inner bootstrapping on training data ---
print("\n" + "="*50)
print("--- REGIME 2: LOOCV with inner bootstrapping on training data ---")
print("="*50 + "\n")

# Helper functions for Regime 2 specific sampling
def safe_sample(group, frac):
    """Safely samples from a pandas DataFrame group."""
    if group.empty:
        return np.empty((0, 3)) # Return empty numpy array if group is empty
    num_original_points = len(group)
    num_points_to_sample = max(1, int(np.floor(num_original_points * frac)))
    # Using replace=True as per original bootstrap logic
    return group.sample(n=num_points_to_sample, replace=True).to_numpy()

def sample_diagrams_for_phase(stacked_phase_data, sampling_fraction, num_subsamples=50):
    """
    Generates bootstrapped diagrams for a single stacked phase.
    stacked_phase_data: np.array of all points for one phase, all dimensions included.
    """
    subsampled_diagrams = []
    if stacked_phase_data.shape[0] == 0:
        return [np.empty((0,3))] * num_subsamples # Return list of empty arrays if no data

    df_pd = pd.DataFrame(stacked_phase_data, columns=['birth', 'death', 'dimension'])

    for _ in range(num_subsamples):
        current_subsample_points = []
        # Group by dimension within the stacked phase data
        for dim, group in df_pd.groupby('dimension'):
            sampled_dim_points = safe_sample(group, sampling_fraction)
            if sampled_dim_points.shape[0] > 0:
                current_subsample_points.append(sampled_dim_points)

        if current_subsample_points: # If any points were sampled
            subsampled_diagrams.append(np.vstack(current_subsample_points))
        else: # If no points were sampled for this iteration
            subsampled_diagrams.append(np.empty((0, 3)))
    return subsampled_diagrams

# Define common classifiers for Regime 2
regime2_classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear', max_iter=5000),
    'SVM': SVC(random_state=42, max_iter=5000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

num_train_subsamples_per_phase = 50 # 50 bootstrap samples per phase for training
sampling_fractions_per_phase = {
    0: 0.1, # Fraction of original points to sample for Phase 0
    1: 0.1,  # Fraction for Phase 1
    2: 0.1   # Fraction for Phase 2
}

# --- Define scenarios for Regime 2 processing ---
# Each scenario specifies how to filter the raw diagrams
regime2_scenarios = {
    "All Dimensions": {
        "train_filter_func": lambda dgms_list, y_labels, phase: [dgm for dgm, label in zip(dgms_list, y_labels) if label == phase],
        "test_filter_func": lambda dgm: dgm,
    },
    "PH0 Diagrams": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 0),
        "test_filter_func": lambda dgm: filter_diagrams_by_dimension([dgm], 0)[0],
    },
    "PH1 Diagrams": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 1),
        "test_filter_func": lambda dgm: filter_diagrams_by_dimension([dgm], 1)[0],
    },
    "PH2 Diagrams": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 2),
        "test_filter_func": lambda dgm: filter_diagrams_by_dimension([dgm], 2)[0],
    },
    "PH1 NW Quadrant": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 1), lambda x: x < 0, lambda y: y > 0, 1),
        "test_filter_func": lambda dgm: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm], 1), lambda x: x < 0, lambda y: y > 0, 1)[0],
    },
    "PH1 NE Quadrant": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 1), lambda x: x > 0, lambda y: y > 0, 1),
        "test_filter_func": lambda dgm: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm], 1), lambda x: x > 0, lambda y: y > 0, 1)[0],
    },
    "PH2 NE Quadrant": {
        "train_filter_func": lambda dgms_list, y_labels, phase: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm for dgm, label in zip(dgms_list, y_labels) if label == phase], 2), lambda x: x > 0, lambda y: y > 0, 2),
        "test_filter_func": lambda dgm: filter_diagrams_by_quadrant(filter_diagrams_by_dimension([dgm], 2), lambda x: x > 0, lambda y: y > 0, 2)[0],
    },
}

for scenario_name, funcs in regime2_scenarios.items():
    print(f"\n***** Processing {scenario_name} (Regime 2) *****")

    # Reset lists for each scenario
    fold_y_true = []
    # Initialize prediction lists for each classifier and transformation type
    predictions_per_classifier_pi = {name: [] for name in regime2_classifiers.keys()}
    predictions_per_classifier_pl = {name: [] for name in regime2_classifiers.keys()}


    loocv_regime2_scenario = LeaveOneOut()

    for fold_idx, (train_index, test_index) in enumerate(tqdm(loocv_regime2_scenario.split(pds), total=len(pds), desc=f"LOOCV for {scenario_name}")):
        try:
            X_train_list_raw_fold = [pds[ind] for ind in train_index] # Raw PDs for training samples in this fold
            X_test_single_raw_fold = pds[test_index[0]] # Raw PD for the single test sample in this fold
            y_train_raw_fold = y_labels[train_index]
            y_test_single_fold = y_labels[test_index][0]

            # --- Prepare training data for this scenario ---
            train_data_by_phase_scenario_stacked = {} # Stores stacked points for each phase & scenario
            for phase in [0, 1, 2]:
                # Apply scenario-specific filter to training diagrams from individual samples for this phase
                filtered_train_dgms_for_phase = funcs["train_filter_func"](X_train_list_raw_fold, y_train_raw_fold, phase)
                # Stack all points for this phase and scenario
                if filtered_train_dgms_for_phase and any(dgm.shape[0] > 0 for dgm in filtered_train_dgms_for_phase):
                    train_data_by_phase_scenario_stacked[phase] = np.vstack(filtered_train_dgms_for_phase)
                else:
                    train_data_by_phase_scenario_stacked[phase] = np.empty((0, 3))


            X_train_augmented_scenario = []
            y_train_augmented_labels_scenario = []

            for phase in [0, 1, 2]:
                if phase in train_data_by_phase_scenario_stacked:
                    current_phase_samples = sample_diagrams_for_phase(
                        train_data_by_phase_scenario_stacked[phase],
                        sampling_fractions_per_phase.get(phase, 0.1),
                        num_subsamples=num_train_subsamples_per_phase
                    )
                    non_empty_current_phase_samples = [s for s in current_phase_samples if s.shape[0] > 0]
                    X_train_augmented_scenario.extend(non_empty_current_phase_samples)
                    y_train_augmented_labels_scenario.extend([phase] * len(non_empty_current_phase_samples))


            if not X_train_augmented_scenario:
                fold_y_true.append(y_test_single_fold)
                for preds_list in predictions_per_classifier_pi.values(): preds_list.append(-1)
                for preds_list in predictions_per_classifier_pl.values(): preds_list.append(-1)
                gc.collect()
                continue # Skip to next fold


            y_train_augmented_scenario = np.array(y_train_augmented_labels_scenario)

            # --- Prepare test data for this scenario ---
            X_test_single_scenario = funcs["test_filter_func"](X_test_single_raw_fold)

            # --- Pad all diagrams for transformation ---
            all_diagrams_for_transform = X_train_augmented_scenario + [X_test_single_scenario]
            padded_diagrams_for_transform = pad_diagrams(all_diagrams_for_transform) # *** Changed to pad_diagrams ***


            if padded_diagrams_for_transform.shape[0] == 0: # If all diagrams became empty after padding for this scenario
                fold_y_true.append(y_test_single_fold)
                for preds_list in predictions_per_classifier_pi.values(): preds_list.append(-1)
                for preds_list in predictions_per_classifier_pl.values(): preds_list.append(-1)
                gc.collect()
                continue

            # --- Transform using GTDA PersistenceImage and PersistenceLandscape ---
            pi_transform_regime2 = PersistenceImage(sigma=1.0, n_bins=100, weight_function=None, n_jobs=1)
            pl_transform_regime2 = PersistenceLandscape(n_layers=5, n_bins=100, n_jobs=1)

            try:
                PIs_transformed_scenario = pi_transform_regime2.fit_transform(padded_diagrams_for_transform)
                PLs_transformed_scenario = pl_transform_regime2.fit_transform(padded_diagrams_for_transform)
            except Exception as e:
                print(f"Error transforming diagrams in fold {fold_idx+1} for {scenario_name}: {e}. Skipping predictions for this fold.")
                fold_y_true.append(y_test_single_fold)
                for preds_list in predictions_per_classifier_pi.values(): preds_list.append(-1)
                for preds_list in predictions_per_classifier_pl.values(): preds_list.append(-1)
                gc.collect()
                continue


            # Reshape to 2D (num_samples, num_features)
            # Check for empty transformed arrays
            if PIs_transformed_scenario.size == 0 or PLs_transformed_scenario.size == 0:
                print(f"Warning: Transformed arrays are empty in fold {fold_idx+1} for {scenario_name}. Skipping predictions.")
                fold_y_true.append(y_test_single_fold)
                for preds_list in predictions_per_classifier_pi.values(): preds_list.append(-1)
                for preds_list in predictions_per_classifier_pl.values(): preds_list.append(-1)
                gc.collect()
                continue

            PIs_transformed_scenario = PIs_transformed_scenario.reshape(PIs_transformed_scenario.shape[0], -1)
            PLs_transformed_scenario = PLs_transformed_scenario.reshape(PLs_transformed_scenario.shape[0], -1)

            # Split into training and test feature sets
            X_train_PI_scenario = PIs_transformed_scenario[:-1]
            X_test_PI_scenario = PIs_transformed_scenario[-1].reshape(1, -1)

            X_train_PL_scenario = PLs_transformed_scenario[:-1]
            X_test_PL_scenario = PLs_transformed_scenario[-1].reshape(1, -1)

            # Store true label for this fold
            fold_y_true.append(y_test_single_fold) # Append the actual label once

            # Train and predict with various classifiers
            for name, clf in regime2_classifiers.items():
                # For PI
                try:
                    clf.fit(X_train_PI_scenario, y_train_augmented_scenario)
                    predictions_per_classifier_pi[name].append(clf.predict(X_test_PI_scenario)[0])
                except Exception as e:
                    predictions_per_classifier_pi[name].append(-1)

                # For PL
                try:
                    clf.fit(X_train_PL_scenario, y_train_augmented_scenario)
                    predictions_per_classifier_pl[name].append(clf.predict(X_test_PL_scenario)[0])
                except Exception as e:
                    predictions_per_classifier_pl[name].append(-1)

            gc.collect() # Explicitly run garbage collector

        except Exception as e:
            print(f"\n Fold {fold_idx+1} for {scenario_name} failed: {e}. Skipping this fold.")
            fold_y_true.append(y_labels[test_index][0])
            for preds_list in predictions_per_classifier_pi.values(): preds_list.append(-1)
            for preds_list in predictions_per_classifier_pl.values(): preds_list.append(-1)
            gc.collect()


    # --- Print results for this scenario in REGIME 2 ---
    y_true_scenario = np.array(fold_y_true)

    # Filter out failed predictions (marked with -1) - check PI Logistic Regression as a proxy
    # Need to check if there are any valid predictions at all before proceeding.
    # Check if the list itself is empty first.
    if not predictions_per_classifier_pi['Logistic Regression']:
        print(f"\n--- No predictions to evaluate for {scenario_name} (Regime 2) ---")
        continue

    valid_indices = (np.array(predictions_per_classifier_pi['Logistic Regression']) != -1)
    if not valid_indices.all():
        print(f"\nWarning: {np.sum(~valid_indices)} folds failed for {scenario_name} and will be excluded from final metrics.")
    else:
        print(f"\nAll folds completed for {scenario_name}.")


    y_true_scenario_filtered = y_true_scenario[valid_indices]

    def print_final_metrics(name, y_true, y_pred):
        if len(y_true) == 0:
            print(f"{name}\n  No valid predictions to evaluate for this scenario.\n")
            return

        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0
        )
        cm = confusion_matrix(y_true, y_pred)

        print(f"{name}")
        print(f"  Accuracy : {acc:.3f}")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall   : {recall:.3f}")
        print(f"  F1 Score : {f1:.3f}")
        print("  Confusion Matrix:\n", cm)
        print("\n")

    print(f"\n--- Final Evaluation Results for {scenario_name} (Regime 2 - Persistence Images) ---")
    for name in regime2_classifiers.keys():
        y_pred_filtered = np.array(predictions_per_classifier_pi[name])[valid_indices].flatten()
        print_final_metrics(name + " (PI)", y_true_scenario_filtered, y_pred_filtered)

    print(f"\n--- Final Evaluation Results for {scenario_name} (Regime 2 - Persistence Landscapes) ---")
    for name in regime2_classifiers.keys():
        y_pred_filtered = np.array(predictions_per_classifier_pl[name])[valid_indices].flatten()
        print_final_metrics(name + " (PL)", y_true_scenario_filtered, y_pred_filtered)



--- REGIME 2: LOOCV with inner bootstrapping on training data ---


***** Processing All Dimensions (Regime 2) *****


LOOCV for All Dimensions: 100%|██████████| 27/27 [14:32<00:00, 32.31s/it]



All folds completed for All Dimensions.

--- Final Evaluation Results for All Dimensions (Regime 2 - Persistence Images) ---
Logistic Regression (PI)
  Accuracy : 0.704
  Precision: 0.848
  Recall   : 0.704
  F1 Score : 0.732
  Confusion Matrix:
 [[4 0 0]
 [5 6 0]
 [2 1 9]]


SVM (PI)
  Accuracy : 0.667
  Precision: 0.669
  Recall   : 0.667
  F1 Score : 0.667
  Confusion Matrix:
 [[0 3 1]
 [1 9 1]
 [3 0 9]]


Random Forest (PI)
  Accuracy : 0.704
  Precision: 0.703
  Recall   : 0.704
  F1 Score : 0.702
  Confusion Matrix:
 [[ 0  2  2]
 [ 1 10  0]
 [ 3  0  9]]


KNN (PI)
  Accuracy : 0.667
  Precision: 0.669
  Recall   : 0.667
  F1 Score : 0.667
  Confusion Matrix:
 [[0 3 1]
 [1 9 1]
 [3 0 9]]



--- Final Evaluation Results for All Dimensions (Regime 2 - Persistence Landscapes) ---
Logistic Regression (PL)
  Accuracy : 0.593
  Precision: 0.563
  Recall   : 0.593
  F1 Score : 0.552
  Confusion Matrix:
 [[ 0  2  2]
 [ 1  5  5]
 [ 1  0 11]]


SVM (PL)
  Accuracy : 0.333
  Precision: 0.36

LOOCV for PH0 Diagrams: 100%|██████████| 27/27 [02:46<00:00,  6.15s/it]



All folds completed for PH0 Diagrams.

--- Final Evaluation Results for PH0 Diagrams (Regime 2 - Persistence Images) ---
Logistic Regression (PI)
  Accuracy : 0.630
  Precision: 0.720
  Recall   : 0.630
  F1 Score : 0.640
  Confusion Matrix:
 [[4 0 0]
 [4 6 1]
 [2 3 7]]


SVM (PI)
  Accuracy : 0.519
  Precision: 0.518
  Recall   : 0.519
  F1 Score : 0.517
  Confusion Matrix:
 [[0 3 1]
 [1 6 4]
 [3 1 8]]


Random Forest (PI)
  Accuracy : 0.519
  Precision: 0.520
  Recall   : 0.519
  F1 Score : 0.519
  Confusion Matrix:
 [[0 3 1]
 [1 7 3]
 [3 2 7]]


KNN (PI)
  Accuracy : 0.519
  Precision: 0.518
  Recall   : 0.519
  F1 Score : 0.517
  Confusion Matrix:
 [[0 3 1]
 [1 6 4]
 [3 1 8]]



--- Final Evaluation Results for PH0 Diagrams (Regime 2 - Persistence Landscapes) ---
Logistic Regression (PL)
  Accuracy : 0.519
  Precision: 0.517
  Recall   : 0.519
  F1 Score : 0.517
  Confusion Matrix:
 [[2 1 1]
 [1 5 5]
 [1 4 7]]


SVM (PL)
  Accuracy : 0.481
  Precision: 0.487
  Recall   : 0.481
  F

...


All folds completed for PH1 NW Quadrant.

--- Final Evaluation Results for PH1 NW Quadrant (Regime 2 - Persistence Images) ---
Logistic Regression (PI)
  Accuracy : 0.778
  Precision: 0.872
  Recall   : 0.778
  F1 Score : 0.798
  Confusion Matrix:
 [[4 0 0]
 [3 8 0]
 [2 1 9]]


SVM (PI)
  Accuracy : 0.519
  Precision: 0.587
  Recall   : 0.519
  F1 Score : 0.523
  Confusion Matrix:
 [[ 0  3  1]
 [ 0 10  1]
 [ 7  1  4]]


Random Forest (PI)
  Accuracy : 0.593
  Precision: 0.672
  Recall   : 0.593
  F1 Score : 0.607
  Confusion Matrix:
 [[ 0  3  1]
 [ 1 10  0]
 [ 5  1  6]]


KNN (PI)
  Accuracy : 0.519
  Precision: 0.587
  Recall   : 0.519
  F1 Score : 0.523
  Confusion Matrix:
 [[ 0  3  1]
 [ 0 10  1]
 [ 7  1  4]]



--- Final Evaluation Results for PH1 NW Quadrant (Regime 2 - Persistence Landscapes) ---
Logistic Regression (PL)
  Accuracy : 0.407
  Precision: 0.449
  Recall   : 0.407
  F1 Score : 0.426
  Confusion Matrix:
 [[0 1 3]
 [3 5 3]
 [3 3 6]]


SVM (PL)
  Accuracy : 0.259
  Pre

LOOCV for PH1 NE Quadrant: 100%|██████████| 27/27 [03:03<00:00,  6.80s/it]



All folds completed for PH1 NE Quadrant.

--- Final Evaluation Results for PH1 NE Quadrant (Regime 2 - Persistence Images) ---
Logistic Regression (PI)
  Accuracy : 0.778
  Precision: 0.877
  Recall   : 0.778
  F1 Score : 0.796
  Confusion Matrix:
 [[4 0 0]
 [2 9 0]
 [3 1 8]]


SVM (PI)
  Accuracy : 0.704
  Precision: 0.668
  Recall   : 0.704
  F1 Score : 0.675
  Confusion Matrix:
 [[ 0  1  3]
 [ 1  8  2]
 [ 1  0 11]]


Random Forest (PI)
  Accuracy : 0.667
  Precision: 0.682
  Recall   : 0.667
  F1 Score : 0.669
  Confusion Matrix:
 [[0 3 1]
 [2 9 0]
 [2 1 9]]


KNN (PI)
  Accuracy : 0.704
  Precision: 0.668
  Recall   : 0.704
  F1 Score : 0.675
  Confusion Matrix:
 [[ 0  1  3]
 [ 1  8  2]
 [ 1  0 11]]



--- Final Evaluation Results for PH1 NE Quadrant (Regime 2 - Persistence Landscapes) ---
Logistic Regression (PL)
  Accuracy : 0.519
  Precision: 0.502
  Recall   : 0.519
  F1 Score : 0.508
  Confusion Matrix:
 [[0 2 2]
 [2 7 2]
 [1 4 7]]


SVM (PL)
  Accuracy : 0.444
  Precision: 0

LOOCV for PH2 NE Quadrant: 100%|██████████| 27/27 [03:17<00:00,  7.33s/it]


All folds completed for PH2 NE Quadrant.

--- Final Evaluation Results for PH2 NE Quadrant (Regime 2 - Persistence Images) ---
Logistic Regression (PI)
  Accuracy : 0.741
  Precision: 0.906
  Recall   : 0.741
  F1 Score : 0.777
  Confusion Matrix:
 [[4 0 0]
 [4 7 0]
 [3 0 9]]


SVM (PI)
  Accuracy : 0.667
  Precision: 0.695
  Recall   : 0.667
  F1 Score : 0.677
  Confusion Matrix:
 [[ 0  2  2]
 [ 1 10  0]
 [ 4  0  8]]


Random Forest (PI)
  Accuracy : 0.667
  Precision: 0.714
  Recall   : 0.667
  F1 Score : 0.688
  Confusion Matrix:
 [[1 2 1]
 [2 8 1]
 [3 0 9]]


KNN (PI)
  Accuracy : 0.667
  Precision: 0.695
  Recall   : 0.667
  F1 Score : 0.677
  Confusion Matrix:
 [[ 0  2  2]
 [ 1 10  0]
 [ 4  0  8]]



--- Final Evaluation Results for PH2 NE Quadrant (Regime 2 - Persistence Landscapes) ---
Logistic Regression (PL)
  Accuracy : 0.556
  Precision: 0.578
  Recall   : 0.556
  F1 Score : 0.566
  Confusion Matrix:
 [[1 3 0]
 [4 5 2]
 [0 3 9]]


SVM (PL)
  Accuracy : 0.444
  Precision: 0